In [7]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [35]:
adult = pd.read_csv("../data/adult.csv", index_col=0)
adult_clean = adult[~adult.isin(["?"]).any(axis=1)].copy()
adult_clean["salary"].value_counts()

<=50K    22654
>50K      7508
Name: salary, dtype: int64

# Task 1
Print the count of men and women in the dataset.

In [36]:
adult_clean["sex"].value_counts()

Male      20380
Female     9782
Name: sex, dtype: int64

# Task 2
Find the average age of men in dataset

In [37]:
average_age_men = adult_clean[adult_clean["sex"] == "Male"]["age"].mean()
round(average_age_men, 2)

39.18

# Task 3
Get the percentage of people from Poland (native-country)

In [38]:
adult_clean["native-country"].value_counts(normalize=True)["Poland"] * 100

0.18566408063125786

# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [39]:
high_income = adult_clean[adult_clean["salary"] == ">50K"]["age"]
mean_high = high_income.mean()
std_high = high_income.std()

low_income = adult_clean[adult_clean["salary"] == "<=50K"]["age"]
mean_low = low_income.mean()
std_low = low_income.std()

round(mean_high, 2), round(std_high, 2), round(mean_low, 2), round(std_low, 2)

(43.96, 10.27, 36.61, 13.46)

In [40]:
age_stats = adult_clean.groupby("salary")["age"].agg(["mean", "std"])
round(age_stats, 2)

,mean,std
salary,,
<=50K,36.61,13.46
>50K,43.96,10.27


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [41]:
higher_education = ["Bachelors", "Prof-school", "Assoc-acdm", "Assoc-voc", "Masters", "Doctorate"]
no_higher_edu = ~adult_clean["education"].isin(higher_education)
high_salary_no_edu = adult_clean[no_higher_edu & (adult_clean["salary"] == ">50K")]
high_salary_no_edu.shape[0]

3178

In [42]:
high_salary_no_edu["education"].count()

3178

# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [43]:
age_stats_by_education = adult_clean.groupby("education")["age"].describe()
age_stats_by_education

,count,mean,std,min,25%,50%,75%,max
education,,,,,,,,
10th,820.0,37.897561,16.225795,17.0,23.0,36.0,52.0,90.0
11th,1048.0,32.363550,15.089307,17.0,18.0,28.5,43.0,90.0
12th,377.0,32.013263,14.373710,17.0,19.0,28.0,41.0,79.0
1st-4th,151.0,44.622517,14.929051,19.0,33.0,44.0,56.0,81.0
5th-6th,288.0,41.649306,14.754622,17.0,28.0,41.0,53.0,82.0
7th-8th,557.0,47.631957,15.737479,17.0,34.0,49.0,60.0,90.0
9th,455.0,40.303297,15.335754,17.0,28.0,38.0,53.0,90.0
Assoc-acdm,1008.0,37.286706,10.509755,19.0,29.0,36.0,44.0,90.0
Assoc-voc,1307.0,38.246366,11.181253,19.0,30.0,37.0,45.0,84.0


# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [54]:
men = adult_clean["sex"] == "Male"
married_men = adult_clean["marital-status"].str.startswith("Married")
married_men_salary = adult_clean[men & married_men]["salary"].value_counts(normalize=True) * 100
non_married_men_salary = adult_clean[men & ~married_men]["salary"].value_counts(normalize=True) * 100
married_men_salary, non_married_men_salary

(<=50K    55.201566
 >50K     44.798434
 Name: salary, dtype: float64,
 <=50K    91.150559
 >50K      8.849441
 Name: salary, dtype: float64)

In [47]:
adult_clean["married"] = adult_clean["marital-status"].str.startswith("Married")
salary_dist = adult_clean[men].groupby("married")["salary"].value_counts(normalize=True) * 100
salary_dist

married  salary
False    <=50K     91.150559
         >50K       8.849441
True     <=50K     55.201566
         >50K      44.798434
Name: salary, dtype: float64

# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [56]:
max_hours = adult_clean["hours-per-week"].max()
num_people = (adult_clean["hours-per-week"] == max_hours).sum()
max_hours, num_people

(99, 78)

# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [58]:
numeric_cols = adult_clean.select_dtypes(include=np.number)
corr_matrix = numeric_cols.corr()
corr_matrix

,age,hours-per-week,salary K$
age,1.000000,0.101599,0.208203
hours-per-week,0.101599,1.000000,0.196378
salary K$,0.208203,0.196378,1.000000
